In [1]:
!pwd

/home/webgr/aston_git/greenplum/artifacts


In [2]:
"""
Загрузка справочников MOEX (securities, boards) и сохранение в Parquet.
"""

import requests
import pandas as pd
from datetime import datetime

MOEX_BASE = "https://iss.moex.com/iss"


In [3]:

def get_securities():
    """Загрузить список ценных бумаг (акции) с биржи MOEX."""
    url = f"{MOEX_BASE}/engines/stock/markets/shares/boards/TQBR/securities.json"
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()
    
    columns = data['securities']['columns']
    rows = data['securities']['data']
    df = pd.DataFrame(rows, columns=columns)
    
    # Оставляем только нужные поля
    required_cols = ['SECID', 'BOARDID', 'ISIN', 'ISQUALIFIEDINVESTORS']
    # Проверяем наличие всех полей
    for col in required_cols:
        if col not in df.columns:
            print(f"Предупреждение: поле {col} отсутствует в ответе")
    # Выбираем существующие
    available = [c for c in required_cols if c in df.columns]
    df = df[available]
    
    # Приводим ISQUALIFIEDINVESTORS к bool (если есть)
    if 'ISQUALIFIEDINVESTORS' in df.columns:
        df['ISQUALIFIEDINVESTORS'] = df['ISQUALIFIEDINVESTORS'].astype(bool)
    
    return df


In [4]:

def get_boards():
    """Загрузить список режимов торгов (boards)."""
    url = f"{MOEX_BASE}/engines/stock/markets/shares/boards.json"
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()
    
    columns = data['boards']['columns']
    rows = data['boards']['data']
    df = pd.DataFrame(rows, columns=columns)
    
    # Оставляем только нужные поля
    required_cols = ['BOARDID', 'BOARD_NAME']
    available = [c for c in required_cols if c in df.columns]
    df = df[available]
    
    return df


In [8]:

def save_to_parquet(df, filename, output_dir='.'):
    """Сохранить DataFrame в Parquet."""
    if df.empty:
        print(f"Предупреждение: DataFrame пуст, файл {filename} не будет создан")
        return
    # os.makedirs(output_dir, exist_ok=True)
    filepath = filename
    # Приводим типы, чтобы избежать проблем с PyArrow
    # for col in df.select_dtypes(include=['period']).columns:
    #     df[col] = df[col].dt.to_timestamp()
    df.to_parquet(filepath, index=False, engine='pyarrow')
    print(f"Сохранено: {filepath} ({len(df)} записей)")


In [6]:

print("Загрузка списка ценных бумаг...")
df_sec = get_securities()


Загрузка списка ценных бумаг...
Предупреждение: поле ISQUALIFIEDINVESTORS отсутствует в ответе


In [9]:

save_to_parquet(df_sec, 'securities.parquet')


Сохранено: securities.parquet (262 записей)


In [10]:

print("Загрузка списка режимов торгов...")
df_boards = get_boards()


Загрузка списка режимов торгов...


In [11]:

save_to_parquet(df_boards, 'boards.parquet')


Предупреждение: DataFrame пуст, файл boards.parquet не будет создан


In [ ]:

print("Готово.")
